# 🚀 RapidSegment Experiment Suite

A **no-code / low-code** experiment workbench that runs entirely inside this Jupyter notebook using only **ipywidgets**.

### Capabilities
| Feature | What it does |
|---------|--------------|
| **Named experiments** | Clear name + notes + tags |
| **Data sources** | Local file (CSV / Parquet / Arrow / Excel) **or** BigQuery |
| **Auto-profile** | Rows, columns, event rate, null rates, target balance |
| **Visual parameter UI** | Every important `StrategicSegmentBuilder` knob |
| **Parameter presets** | Balanced / High-lift / Volume-first / Fast exploratory |
| **Experiment queue** | Schedule many runs; they execute **sequentially** with progress bar + live status |
| **Full reproducibility** | Config, stats, segments, coverage, logs, environment metadata persisted on disk |
| **Clone** | Duplicate any past experiment in one click |
| **Results explorer** | Segment table + coverage summary (same spirit as `evaluate_final_coverage`) + log tail |
| **Compare** | Side-by-side coverage of two experiments |
| **Export ZIP** | Archive a full experiment folder for sharing / audit |

> **Requirements**: `rapidsegment`, `ipywidgets`, `pandas`, `duckdb`, `pyarrow`  
> Optional: `openpyxl` (Excel), `google-cloud-bigquery` (BigQuery)


## 0 · Environment check

In [ ]:
# Optional installs (uncomment as needed)
# %pip install -q rapidsegment ipywidgets pandas duckdb pyarrow
# %pip install -q openpyxl
# %pip install -q "google-cloud-bigquery[pandas]"

import sys, importlib
print(f"Python {sys.version.split()[0]}")
for pkg in ["rapidsegment", "ipywidgets", "pandas", "duckdb", "pyarrow"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  ✅ {pkg} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"  ❌ {pkg}  →  please install")


## 1 · Load the suite

The implementation lives in `rs_experiment_suite.py` (same folder as this notebook).  
Make sure that file is on your Python path (or sits next to this notebook).


In [ ]:
import sys
from pathlib import Path

# Ensure the directory containing this notebook is importable
nb_dir = Path.cwd()
if str(nb_dir) not in sys.path:
    sys.path.insert(0, str(nb_dir))

from rs_experiment_suite import launch_suite, SUITE_ROOT
print(f"Suite ready. Experiments will be stored under:\n  {SUITE_ROOT}")


## 2 · Launch the UI

In [ ]:
suite = launch_suite()
# Keep this notebook open. All artefacts land in ./rs_experiments/


## 3 · Optional – synthetic demo data

Run this once if you want to try the suite without your own dataset.  
Then in the UI set **Source = Local file**, path = `demo_rapidsegment.parquet`,  
**Target col** = `default_flag`, **Primary key** = `cust_id`.


In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 40_000
demo = pd.DataFrame({
    "cust_id": [f"C{i:06d}" for i in range(n)],
    "max_dpd_12m": np.random.choice([0, 15, 30, 60, 90], n, p=[0.65, 0.15, 0.1, 0.07, 0.03]),
    "utilization_avg_3m": np.round(np.random.beta(2, 5, n) * 1.3, 3),
    "income_band": np.random.choice(["L", "M", "H"], n, p=[0.4, 0.4, 0.2]),
    "tenure_m": np.random.randint(1, 120, n),
    "product": np.random.choice(["A", "B", "C"], n),
    "region": np.random.choice(["N", "S", "E", "W"], n),
    "default_flag": np.random.binomial(1, 0.06, n),
})
mask = (demo["max_dpd_12m"] >= 60) & (demo["utilization_avg_3m"] >= 0.7)
demo.loc[mask, "default_flag"] = np.random.binomial(1, 0.55, mask.sum())

path = "demo_rapidsegment.parquet"
demo.to_parquet(path, index=False)
print(f"Wrote {path}  ({len(demo):,} rows)")
print("Target = default_flag | Primary key = cust_id")
print("Base event rate:", round(demo["default_flag"].mean() * 100, 2), "%")


## 4 · How to use

1. **Experiments tab** – browse past runs; Load / Clone / Delete / Add to queue.  
2. **Setup tab**  
   - Name, notes, tags  
   - Choose **Local file** or **BigQuery** and fill the fields  
   - Set **Target col** (required)  
   - Click **Load & Profile data** → rows, event rate, null rates appear  
   - Tune parameters (or apply a **Preset**)  
   - **Save experiment** or **Save & add to queue**  
3. **Queue & Run tab** – reorder items, then **Run queue sequentially**.  
   Progress bar + live status + scrolling log. Experiments run one after another.  
   You can stop after the current experiment finishes.  
4. **Results tab** – pick an experiment → **Show results**  
   (segments table, coverage, log tail).  
   Compare two experiments or export a ZIP.

### What gets stored (reproducibility)

Each experiment folder under `./rs_experiments/<exp_id>/` contains:

| File | Content |
|------|---------|
| `metadata.json` | Name, status, timestamps, env versions, error if any |
| `data_source.json` | Exact source configuration |
| `builder_params.json` | Full parameter snapshot |
| `stats.json` | Profile (rows, nulls, event rate…) |
| `segments.json` | Extracted rules + metrics |
| `coverage.json` | Hierarchical evaluation on the original data |
| `diagnostics.json` | Per-iteration feature journey (when available) |
| `logs.txt` / `logs.json` | Full run log |

### Suggested workflows
- **Clone → tweak one knob → re-queue** for clean A/B on settings.  
- Use **tags** (`prod`, `explore`, `high-lift`) to organise runs.  
- **Export ZIP** before cleaning the folder so the exact artefact travels with your report.  
- Start with the synthetic demo to verify the suite end-to-end.
